In [2]:
import os
import json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

tf.random.set_seed(42)
np.random.seed(42)

print("TensorFlow:", tf.__version__)


TensorFlow: 2.10.0


In [3]:
# GPU Configuration
print("=" * 70)
print("GPU SETUP")
print("=" * 70)

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✓ GPU DETECTED: {len(gpus)} GPU(s) available")
    for gpu in gpus:
        print(f"  {gpu}")
        # Enable memory growth to prevent OOM errors
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("⚠ NO GPU DETECTED - Training will be slow on CPU")

print(f"\nTensorFlow version: {tf.__version__}")
print("=" * 70)

GPU SETUP
✓ GPU DETECTED: 1 GPU(s) available
  PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')

TensorFlow version: 2.10.0


In [5]:
train_dir = "asl_alphabet_train"  
test_dir  = "asl_alphabet_test"

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
NUM_CLASSES = 29

In [6]:
# Training generator (augmentation ON)
train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    validation_split=0.2
)

# Validation generator (NO augmentation)
val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    validation_split=0.2
)

train_gen = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="training",
    shuffle=True,
    seed=42
)

val_gen = val_datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="categorical",
    subset="validation",
    shuffle=False,
    seed=42,
    classes=list(train_gen.class_indices.keys())
)

class_names = list(train_gen.class_indices.keys())

with open("class_names.json", "w") as f:
    json.dump(class_names, f)

print("Classes:", class_names)


Found 69600 images belonging to 29 classes.
Found 17400 images belonging to 29 classes.
Classes: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'del', 'nothing', 'space']


In [7]:
base_model = EfficientNetB0(
    include_top=False,
    weights="imagenet",
    input_shape=(224, 224, 3)
)

base_model.trainable = False  

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation="relu")(x)
x = Dropout(0.4)(x)
output = Dense(NUM_CLASSES, activation="softmax")(x)

model = Model(base_model.input, output)

model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 224, 224, 3  0           []                               
                                )]                                                                
                                                                                                  
 rescaling (Rescaling)          (None, 224, 224, 3)  0           ['input_1[0][0]']                
                                                                                                  
 normalization (Normalization)  (None, 224, 224, 3)  7           ['rescaling[0][0]']              
                                                                                                  
 rescaling_1 (Rescaling)        (None, 224, 224, 3)  0           ['normalization[0][0]']      

In [ ]:
callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2
    ),
    ModelCheckpoint(
    "best_asl_weights.h5",
    monitor="val_accuracy",
    save_best_only=True,
    save_weights_only=True,  
    mode="max",
    verbose=1
)
]

In [11]:
history = model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=15,
    callbacks=callbacks
)

Epoch 1/15
4350/4350 [==============================] - ETA: 0s - loss: 0.2177 - accuracy: 0.9455
Epoch 1: val_accuracy improved from -inf to 0.91402, saving model to best_asl_weights.h5
4350/4350 [==============================] - 804s 185ms/step - loss: 0.2177 - accuracy: 0.9455 - val_loss: 0.2786 - val_accuracy: 0.9140 - lr: 1.0000e-04
Epoch 2/15
4350/4350 [==============================] - ETA: 0s - loss: 0.1264 - accuracy: 0.9680
Epoch 2: val_accuracy improved from 0.91402 to 0.91782, saving model to best_asl_weights.h5
4350/4350 [==============================] - 770s 177ms/step - loss: 0.1264 - accuracy: 0.9680 - val_loss: 0.2435 - val_accuracy: 0.9178 - lr: 1.0000e-04
Epoch 3/15
4350/4350 [==============================] - ETA: 0s - loss: 0.0882 - accuracy: 0.9769
Epoch 3: val_accuracy improved from 0.91782 to 0.92333, saving model to best_asl_weights.h5
4350/4350 [==============================] - 773s 178ms/step - loss: 0.0882 - accuracy: 0.9769 - val_loss: 0.2261 - val_accur

In [12]:
# Load best weights saved by ModelCheckpoint
model.load_weights("best_asl_weights.h5")
print("Best weights loaded successfully")


Best weights loaded successfully


In [13]:
val_loss, val_acc = model.evaluate(val_gen, verbose=1)

print("="*50)
print(f"Validation Accuracy : {val_acc:.4f} ({val_acc*100:.2f}%)")
print(f"Validation Loss     : {val_loss:.4f}")
print("="*50)


1088/1088 [==============================] - 79s 72ms/step - loss: 0.2099 - accuracy: 0.9302
Validation Accuracy : 0.9302 (93.02%)
Validation Loss     : 0.2099


In [3]:
tf.saved_model.save(model, "asl_efficientnet_saved_modelll")
print("Model saved successfully ✅")


INFO:tensorflow:Assets written to: asl_efficientnet_saved_modelll\assets


INFO:tensorflow:Assets written to: asl_efficientnet_saved_modelll\assets


Model saved successfully ✅


In [4]:
model.save("asl_efficientnet_saved_model.h5")
print("Model saved successfully in H5 format ✅")

Model saved successfully in H5 format ✅
